In [17]:
from typing import List,TypedDict
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
docs = (
    PyPDFLoader("documents/Fisher-aware_Quantization.pdf").load()
    + PyPDFLoader("documents/FQ-ViT.pdf").load()
    + PyPDFLoader("documents/Quantization_Variation.pdf").load()
)

In [10]:
len(docs)

51

In [11]:
docs[1]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-07-08T00:05:11+00:00', 'author': 'Huanrui Yang, Yafeng Huang, Zhen Dong, Denis A Gudovskiy, Tomoyuki Okuno, Yohei Nakata, Yuan Du, Kurt Keutzer, Shanghang Zhang', 'keywords': 'DETR, quantization, critical-category, Fisher', 'moddate': '2024-07-08T00:05:11+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': 'Fisher-aware Quantization for DETR Detectors with Critical-category Objectives', 'trapped': '/False', 'source': 'documents/Fisher-aware_Quantization.pdf', 'total_pages': 17, 'page': 1, 'page_label': '2'}, page_content='Fisher-aware Quantization for DETR Detectors with Critical-category Objectives\nFigure 1: Overview. We investigate a practical setting with task-dependent critical-category objectives in Section 3.2. We\nempirically observe disparate effects of quantization on the critical-cate

In [12]:
chunks=RecursiveCharacterTextSplitter(chunk_size=900,chunk_overlap=150).split_documents(documents=docs)
for chunk in chunks:
    chunk.page_content = chunk.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")


In [18]:

embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=os.getenv("HF_API_TOKEN")
)



In [19]:
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [20]:

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)



In [ ]:
class State(TypedDict):
    question:str
    docs:List[str]
    fianl_docs:str